# Lesson 3 — Memory and context

Four moves to keep an agent useful as conversations grow and APIs misbehave.

Covers `s06` + `s08` + `s09` + `s11`. Narration in `speaker_notes/03_memory_and_context.md`.

In [ ]:
import json
import subprocess
import time
from pathlib import Path

from llm_client import get_client, DEFAULT_MODEL, FALLBACK_MODEL

client = get_client()

import platform

IS_WINDOWS = platform.system() == "Windows"
SHELL_HINT = (
    "cmd.exe on Windows — use `dir`, `type`, `cd` (no current dir prints "
    "the path), `where`, `findstr`. Avoid Unix-only commands like `ls`, "
    "`pwd`, `cat`, `grep`, `sleep`, `&&`-chained pipelines."
    if IS_WINDOWS else
    "bash on POSIX — `ls`, `pwd`, `cat`, `grep`, etc."
)

def run_bash(command: str) -> str:
    try:
        out = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        return (out.stdout + out.stderr)[:4000] or "(no output)"
    except Exception as e:
        return f"error: {e}"

BASH_TOOL = {"type": "function", "function": {
    "name": "bash", "description": f"Run a shell command. Shell: {SHELL_HINT}",
    "parameters": {"type": "object",
        "properties": {"command": {"type": "string"}}, "required": ["command"]}}}

## Part 1 — Subagents

A side task with 30 tool calls would pollute the main conversation. Spawn a subagent with fresh `messages=[]`; it runs to completion and returns one summary string.

In [ ]:
SUBAGENT_SYSTEM = (
    "You are a research subagent. Investigate the assigned question using the bash tool. "
    "When you have enough information, reply with a concise summary — no preamble."
)

def spawn_subagent(description: str, *, max_turns=8) -> str:
    """Run a self-contained loop with its own context. Return its final text."""
    messages = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user", "content": description},
    ]
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=DEFAULT_MODEL, messages=messages, tools=[BASH_TOOL]
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content or "(empty)"
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": run_bash(**args)})
    return "(subagent hit max_turns)"

In [ ]:
# Expose subagent-spawning as a tool the main agent can call
SUBAGENT_TOOL = {"type": "function", "function": {
    "name": "spawn_subagent",
    "description": "Run an isolated research subtask. Returns only its summary.",
    "parameters": {"type": "object",
        "properties": {"description": {"type": "string"}},
        "required": ["description"]}}}

summary = spawn_subagent("How many .py files are in this repository? Just give the number.")
print("subagent says:", summary)

The main thread saw only the summary — none of the intermediate tool noise.

## Part 2 — Context compaction

Cheap layers first, expensive last. We implement two of the four real layers: `micro_compact` drops old tool messages; `snip_compact` snips the middle when over budget.

In [ ]:
def approx_tokens(messages) -> int:
    """Rough estimate: 1 token ≈ 4 chars. Good enough for budget decisions."""
    return sum(len(json.dumps(m, default=str)) for m in messages) // 4

KEEP_HEAD = 2   # preserve initial system + first user message
KEEP_TAIL = 6   # preserve the most recent N messages so the model has live context

def snip_compact(messages, limit=4000):
    if approx_tokens(messages) <= limit or len(messages) <= KEEP_HEAD + KEEP_TAIL:
        return messages
    head = messages[:KEEP_HEAD]
    tail = messages[-KEEP_TAIL:]
    snipped = len(messages) - KEEP_HEAD - KEEP_TAIL
    marker = {"role": "user",
              "content": f"[snip: {snipped} earlier messages omitted to save context]"}
    return head + [marker] + tail

def micro_compact(messages, limit=4000):
    """Replace bodies of old tool results with a short placeholder."""
    if approx_tokens(messages) <= limit:
        return messages
    out = []
    cutoff = len(messages) - KEEP_TAIL
    for i, m in enumerate(messages):
        if i < cutoff and m.get("role") == "tool":
            content = str(m.get("content", ""))
            if len(content) > 200:
                m = {**m, "content": f"[compacted {len(content)} chars]"}
        out.append(m)
    return out

In [ ]:
# Demo: build a long synthetic conversation and watch tokens drop
long = [{"role": "system", "content": "system"}, {"role": "user", "content": "start"}]
for i in range(40):
    long.append({"role": "assistant", "content": f"thinking step {i}"})
    long.append({"role": "tool", "tool_call_id": f"t{i}",
                 "content": "x" * 400})

before = approx_tokens(long)
after_micro = approx_tokens(micro_compact(long))
after_snip = approx_tokens(snip_compact(long))
print(f"before: {before} tokens")
print(f"after micro_compact: {after_micro}")
print(f"after snip_compact:  {after_snip}")

## Part 3 — Memory

Memory ≠ conversation history. Memory persists across sessions. Index in the system prompt; bodies on demand — the same pattern as skills.

In [ ]:
MEMORY_DIR = Path(".memory_demo")
MEMORY_DIR.mkdir(exist_ok=True)
INDEX = MEMORY_DIR / "MEMORY.md"
if not INDEX.exists():
    INDEX.write_text("# Memory index\n")

def save_memory(name: str, body: str, description: str):
    path = MEMORY_DIR / f"{name}.md"
    path.write_text(f"---\nname: {name}\ndescription: {description}\n---\n\n{body}\n")
    pointer = f"- [{name}]({name}.md) — {description}\n"
    existing = INDEX.read_text()
    if pointer not in existing:
        INDEX.write_text(existing + pointer)

def read_index() -> str:
    return INDEX.read_text()

def read_memory(name: str) -> str:
    path = MEMORY_DIR / f"{name}.md"
    return path.read_text() if path.exists() else f"no memory: {name}"

# Seed a couple of memories
save_memory("user_role", "User is a data scientist focused on observability.",
            "User profile — affects how to frame explanations")
save_memory("test_framework", "This project uses pytest, not unittest.",
            "Project convention — use when writing or running tests")
print(read_index())

In [ ]:
# Expose memory_read as a tool so the agent can pull in any memory by name
MEMORY_READ_TOOL = {"type": "function", "function": {
    "name": "memory_read",
    "description": "Read a memory by name (without .md). Lookup names from the MEMORY index in your system prompt.",
    "parameters": {"type": "object",
        "properties": {"name": {"type": "string"}}, "required": ["name"]}}}

def build_system_with_memory(base: str) -> str:
    return base + "\n\n# Available memory\n" + read_index()

# A one-turn demo: the agent decides which memory to load.
system = build_system_with_memory(
    "You are an assistant. Before answering questions about the user or project conventions, "
    "call memory_read with the relevant name from the index."
)
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": "Should I write a new test using unittest? Check your memory."},
]
resp = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=[MEMORY_READ_TOOL])
msg = resp.choices[0].message
print("finish_reason:", resp.choices[0].finish_reason)
for c in (msg.tool_calls or []):
    print("  agent called:", c.function.name, c.function.arguments)

The model picked the relevant memory itself — self-organized retrieval.

## Part 4 — Error recovery

Three error classes, three policies: rate limit → backoff; context too long → aggressive compact + retry; other → bounded retries then surface.

In [ ]:
from openai import APIError, RateLimitError, BadRequestError

class RecoveryState:
    def __init__(self):
        self.consecutive_overload = 0
        self.model = DEFAULT_MODEL
        self.has_escalated_tokens = False
        self.has_reactive_compacted = False

def call_with_recovery(messages, tools, state: RecoveryState, *, max_tokens=2048, max_retries=4):
    """Wraps client.chat.completions.create with the three recovery paths."""
    delay = 1.0
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=state.model, messages=messages, tools=tools, max_tokens=max_tokens,
            )
            # Path 1: max_tokens hit — escalate once, then continuation prompt
            if resp.choices[0].finish_reason == "length":
                if not state.has_escalated_tokens:
                    state.has_escalated_tokens = True
                    max_tokens *= 4
                    continue
                messages.append({"role": "user", "content":
                    "Your last response was cut off. Please continue where you left off."})
                continue
            state.consecutive_overload = 0   # success — reset counter
            return resp
        except RateLimitError:
            state.consecutive_overload += 1
            if state.consecutive_overload >= 2 and state.model != FALLBACK_MODEL:
                print(f"  ↪ switching to fallback model {FALLBACK_MODEL}")
                state.model = FALLBACK_MODEL
            time.sleep(delay); delay *= 2
        except BadRequestError as e:
            # Path 2: prompt too long
            if "context" in str(e).lower() and not state.has_reactive_compacted:
                print("  ↪ reactive compaction")
                messages[:] = micro_compact(snip_compact(messages, limit=2000), limit=2000)
                state.has_reactive_compacted = True
                continue
            raise
        except APIError:
            time.sleep(delay); delay *= 2
    raise RuntimeError("call_with_recovery: exhausted retries")

## Putting it together

All four mechanisms in one loop: compact at top, recovery wraps the API call, subagent + memory_read are tools the model decides to use.

In [ ]:
TOOLS = [BASH_TOOL, SUBAGENT_TOOL, MEMORY_READ_TOOL]
HANDLERS = {
    "bash": run_bash,
    "spawn_subagent": spawn_subagent,
    "memory_read": read_memory,
}

def agent_loop(user_prompt: str, *, max_turns=8):
    state = RecoveryState()
    base_system = (
        "You are an agent. Use spawn_subagent for research tasks that need many tool calls. "
        "Use memory_read for facts about the user or project conventions."
    )
    system = build_system_with_memory(base_system)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]

    for _ in range(max_turns):
        # Compact before sending (cheap layers only)
        messages = micro_compact(snip_compact(messages))

        resp = call_with_recovery(messages, TOOLS, state)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            handler = HANDLERS.get(call.function.name)
            output = handler(**args) if handler else f"unknown tool: {call.function.name}"
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(output)})

    return "(max_turns reached)"

In [ ]:
answer = agent_loop(
    "Spawn a subagent to count how many Python files are in this repo. "
    "Then check your memory to recall what test framework we use, and combine into one sentence.",
)
print("\nfinal:", answer)

## Recap

Subagents, compaction, memory, recovery — production stability for one agent.

Lesson 4 adds operational concurrency: task DAG, background threads, cron, worktrees.